# Multi-Agent Supervisor — Route Questions to Specialist Agents
## Domain Dispatch — LLM Classifier and Specialist Agents
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/80-multi-agent-supervisor/multi_agent_supervisor_workbook.ipynb)

A supervisor node classifies each question into math/history/science, then routes to the matching specialist agent. Demonstrates how LangGraph conditional edges implement a supervisor-worker pattern.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Supervisor pattern vs monolithic agent; routing by classification |
| 2 | **Supervisor Node** — LLM outputs one word: math, history, or science |
| 3 | **Specialist Agents** — math_agent, history_agent, science_agent — each with domain system prompt |
| 4 | **Conditional Routing** — lambda s: s["next"] maps domain to the right node |
| 5 | **Full Demo** — 6 questions across all three domains |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

## Part 1 — Supervisor Pattern Concepts

### Why multi-agent?

A single large agent prompted to be good at everything often ends up mediocre at each task. Specialist agents — each with a focused system prompt and, optionally, domain-specific tools — outperform generalists on well-defined subtasks.

The **supervisor** orchestrates the specialists:
1. Receives the user's question
2. Classifies it into a domain
3. Routes it to the correct specialist
4. Returns the specialist's answer

### Routing strategies

| Strategy | How it works | Best for |
|----------|-------------|----------|
| **LLM classification** | Supervisor LLM outputs a domain label | Open-ended routing, many domains |
| **Keyword/regex** | Pattern match on question text | Fast, cheap, when inputs are predictable |
| **Tool-based** | Supervisor calls a `route_to` tool | When the LLM already uses tools |
| **Confidence-based** | Classify + score; fallback if score < threshold | When quality matters more than speed |

### LangGraph conditional edges

```python
g.add_conditional_edges(
    "supervisor",           # source node
    lambda s: s["next"],   # router function — reads state, returns next-node name
    {"math": "math_agent", "history": "history_agent"},  # routing table
)
```

The router function can be arbitrarily complex — it's just a Python function that returns a string.

> **Prerequisite:** Familiar with LangGraph `add_conditional_edges` and `TypedDict` state (example 62).

In [ ]:
from typing import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 'next' is the routing key — supervisor writes it, conditional_edges reads it
class SupervisorState(TypedDict):
    question: str
    next:     str
    answer:   str

def supervisor_node(state: SupervisorState) -> dict:
    # constrained output: one word → more reliable and cheaper than free text
    r = llm.invoke([
        SystemMessage(content="Classify into exactly one word: math, history, or science."),
        HumanMessage(content=state["question"]),
    ])
    domain = r.content.strip().lower()
    print(f"  [supervisor] classified as: {domain!r}")
    return {"next": domain}

SPECIALISTS = {
    "math":    "You are a math expert. Give a concise, accurate answer.",
    "history": "You are a history expert. Give a concise, accurate answer.",
    "science": "You are a science expert. Give a concise, accurate answer.",
}

def make_specialist(domain: str, system_prompt: str):
    def node(state: SupervisorState) -> dict:
        r = llm.invoke([SystemMessage(content=system_prompt),
                        HumanMessage(content=state["question"])])
        return {"answer": r.content.strip()}
    node.__name__ = f"{domain}_agent"
    return node

g = StateGraph(SupervisorState)
g.add_node("supervisor", supervisor_node)
for domain, prompt in SPECIALISTS.items():
    g.add_node(f"{domain}_agent", make_specialist(domain, prompt))
    g.add_edge(f"{domain}_agent", END)

g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", lambda s: s["next"],
    {d: f"{d}_agent" for d in SPECIALISTS})
app = g.compile()

# test the routing
questions = [
    "What is 144 divided by 12?",
    "Who invented the telephone?",
    "What is the speed of light in m/s?",
]
for q in questions:
    r = app.invoke({"question": q, "next": "", "answer": ""})
    print(f"Q: {q}")
    print(f"A: {r['answer'][:100]}")
    print()

## Part 2 — Confidence-Based Routing with Fallback

A supervisor that outputs a single word can hallucinate a domain that doesn't exist (e.g., `"geography"` when you only have three agents). A more robust pattern:

1. Ask the supervisor to output JSON: `{"domain": "math", "confidence": 0.9}`
2. If confidence < threshold, route to a **fallback** general-purpose agent
3. If the domain isn't in the routing table, also fall back

This avoids `KeyError` crashes and produces graceful degradation when the question is ambiguous.

In [ ]:
import json as _json

class ConfidentState(TypedDict):
    question:   str
    next:       str
    confidence: float
    answer:     str

CONFIDENCE_THRESHOLD = 0.7

def supervisor_with_confidence(state: ConfidentState) -> dict:
    r = llm.invoke([
        SystemMessage(content=(
            'Classify the question into one of: math, history, science. '
            'Respond ONLY with JSON: {"domain": "<label>", "confidence": <0-1>}'
        )),
        HumanMessage(content=state["question"]),
    ])
    try:
        parsed = _json.loads(r.content.strip())
        domain = parsed.get("domain", "").lower()
        conf   = float(parsed.get("confidence", 0.0))
    except Exception:
        domain, conf = "unknown", 0.0

    # route to fallback if low confidence or unknown domain
    if conf < CONFIDENCE_THRESHOLD or domain not in SPECIALISTS:
        print(f"  [supervisor] low confidence ({conf:.2f}) for {domain!r} → fallback")
        return {"next": "fallback", "confidence": conf}

    print(f"  [supervisor] {domain!r} confidence={conf:.2f}")
    return {"next": domain, "confidence": conf}

def fallback_agent(state: ConfidentState) -> dict:
    r = llm.invoke([
        SystemMessage(content="You are a general-purpose assistant."),
        HumanMessage(content=state["question"]),
    ])
    return {"answer": f"[fallback] {r.content.strip()}"}

g2 = StateGraph(ConfidentState)
g2.add_node("supervisor", supervisor_with_confidence)
g2.add_node("fallback",   fallback_agent)
for domain, prompt in SPECIALISTS.items():
    specialist_fn = make_specialist(domain, prompt)
    g2.add_node(f"{domain}_agent", specialist_fn)
    g2.add_edge(f"{domain}_agent", END)
g2.add_edge("fallback", END)
g2.add_edge(START, "supervisor")
g2.add_conditional_edges("supervisor", lambda s: s["next"],
    {**{d: f"{d}_agent" for d in SPECIALISTS}, "fallback": "fallback"})
app2 = g2.compile()

test_questions = [
    "What is 256 * 3?",
    "When did World War II end?",
    "What is the capital of France?",   # geography → should hit fallback
    "Tell me a haiku.",                  # off-domain → fallback
]
for q in test_questions:
    r = app2.invoke({"question": q, "next": "", "confidence": 0.0, "answer": ""})
    print(f"Q: {q}")
    print(f"A: {r['answer'][:100]}")
    print()

## Part 3 — Adding a Fourth Specialist and Error Handling

Extending the supervisor to support a new domain is a two-line change:
1. Add the specialist node
2. Add it to the routing table

This shows the architectural advantage of the supervisor pattern: specialists are **pluggable** without touching any other node.

### Error node pattern

Wrap specialist execution in try/except and route to an `error_node` on failure. The error node logs the failure and returns a structured error message.

```python
def safe_specialist(state):
    try:
        return real_specialist(state)
    except Exception as e:
        return {"next": "error", "answer": str(e)}
```

In [ ]:
SPECIALISTS_V2 = {
    **SPECIALISTS,
    "coding": "You are a coding expert. Explain code concepts concisely in plain English.",
}

class SupervisorStateV2(TypedDict):
    question:   str
    next:       str
    confidence: float
    answer:     str
    error:      str

def supervisor_v2(state: SupervisorStateV2) -> dict:
    r = llm.invoke([
        SystemMessage(content=(
            'Classify into one of: math, history, science, coding. '
            'Respond ONLY with JSON: {"domain": "<label>", "confidence": <0-1>}'
        )),
        HumanMessage(content=state["question"]),
    ])
    try:
        parsed = _json.loads(r.content.strip())
        domain = parsed.get("domain", "").lower()
        conf   = float(parsed.get("confidence", 0.0))
    except Exception:
        domain, conf = "unknown", 0.0

    if conf < CONFIDENCE_THRESHOLD or domain not in SPECIALISTS_V2:
        return {"next": "fallback", "confidence": conf}
    return {"next": domain, "confidence": conf}

def make_safe_specialist(domain: str, prompt: str):
    def node(state: SupervisorStateV2) -> dict:
        try:
            r = llm.invoke([SystemMessage(content=prompt),
                            HumanMessage(content=state["question"])])
            return {"answer": r.content.strip(), "error": ""}
        except Exception as exc:
            return {"answer": "", "error": str(exc), "next": "error_node"}
    node.__name__ = f"{domain}_agent"
    return node

def error_node(state: SupervisorStateV2) -> dict:
    return {"answer": f"[error] Agent failed: {state['error']}"}

def fallback_v2(state: SupervisorStateV2) -> dict:
    r = llm.invoke([SystemMessage(content="You are a helpful assistant."),
                    HumanMessage(content=state["question"])])
    return {"answer": f"[fallback] {r.content.strip()}"}

g3 = StateGraph(SupervisorStateV2)
g3.add_node("supervisor", supervisor_v2)
g3.add_node("fallback",   fallback_v2)
g3.add_node("error_node", error_node)
for domain, prompt in SPECIALISTS_V2.items():
    g3.add_node(f"{domain}_agent", make_safe_specialist(domain, prompt))
    g3.add_edge(f"{domain}_agent", END)
g3.add_edge("fallback",   END)
g3.add_edge("error_node", END)
g3.add_edge(START, "supervisor")
g3.add_conditional_edges("supervisor", lambda s: s["next"],
    {**{d: f"{d}_agent" for d in SPECIALISTS_V2}, "fallback": "fallback"})
app3 = g3.compile()

coding_questions = [
    "What does a for loop do?",
    "Explain what a hash map is.",
    "What is recursion?",
    "What year did World War II end?",
]
for q in coding_questions:
    r = app3.invoke({"question": q, "next": "", "confidence": 0.0,
                     "answer": "", "error": ""})
    print(f"Q: {q}")
    print(f"A: {r['answer'][:120]}")
    print()

## Part 4 — Parallel Specialists and Result Aggregation

Some questions span multiple domains. Instead of routing to *one* specialist, you can fan out to *all* of them and aggregate:

```
START → supervisor → [math_agent, science_agent] → aggregate_node → END
```

LangGraph supports parallel branches natively — add `START → node_a` and `START → node_b` edges and both nodes run concurrently (or in parallel threads, depending on the executor).

The `aggregate_node` receives all partial results in state and synthesizes a final answer.

This pattern is useful for:
- Synthesis questions that cut across domains
- Confidence-weighted voting across specialists
- Finding the "best" answer when you're uncertain which domain applies

In [ ]:
# Parallel specialist fan-out
class FanoutState(TypedDict):
    question:      str
    answer_math:   str
    answer_history:str
    answer_science:str
    final_answer:  str

def math_fan(state: FanoutState) -> dict:
    r = llm.invoke([SystemMessage(content=SPECIALISTS["math"]),
                    HumanMessage(content=state["question"])])
    return {"answer_math": r.content.strip()}

def history_fan(state: FanoutState) -> dict:
    r = llm.invoke([SystemMessage(content=SPECIALISTS["history"]),
                    HumanMessage(content=state["question"])])
    return {"answer_history": r.content.strip()}

def science_fan(state: FanoutState) -> dict:
    r = llm.invoke([SystemMessage(content=SPECIALISTS["science"]),
                    HumanMessage(content=state["question"])])
    return {"answer_science": r.content.strip()}

def aggregate_node(state: FanoutState) -> dict:
    candidates = [
        ("math",    state["answer_math"]),
        ("history", state["answer_history"]),
        ("science", state["answer_science"]),
    ]
    # ask the LLM to pick the best answer and explain
    prompt = (
        f"Question: {state['question']}\n\n"
        "Candidate answers from three specialists:\n" +
        "\n".join(f"  {d}: {a[:200]}" for d, a in candidates if a) +
        "\n\nPick the most accurate and relevant answer. Respond with: 'Best: <text>'"
    )
    r = llm.invoke([SystemMessage(content="You are an answer arbiter."),
                    HumanMessage(content=prompt)])
    return {"final_answer": r.content.strip()}

gf = StateGraph(FanoutState)
for name, fn in [("math_fan", math_fan), ("history_fan", history_fan),
                  ("science_fan", science_fan), ("aggregate", aggregate_node)]:
    gf.add_node(name, fn)

# fan-out: all three specialists start simultaneously
gf.add_edge(START,         "math_fan")
gf.add_edge(START,         "history_fan")
gf.add_edge(START,         "science_fan")
# fan-in: aggregate waits for all three
gf.add_edge("math_fan",    "aggregate")
gf.add_edge("history_fan", "aggregate")
gf.add_edge("science_fan", "aggregate")
gf.add_edge("aggregate",   END)
fanout_app = gf.compile()

cross_domain_questions = [
    "What is the relationship between math and music?",
    "How did Newtonian physics change history?",
]
for q in cross_domain_questions:
    r = fanout_app.invoke({"question": q,
                           "answer_math": "", "answer_history": "",
                           "answer_science": "", "final_answer": ""})
    print(f"Q: {q}")
    print(f"Final: {r['final_answer'][:200]}")
    print()

## Exercises

---

**Exercise 1 — Add a Creative Writing Specialist**

Add a `creative_agent` that answers with a poem or metaphor. Extend the supervisor prompt to classify `"creative"` as a domain. Test with: `"Explain gravity as a poem"`.

---

**Exercise 2 — Confidence Voting**

Modify the parallel fan-out so each specialist also outputs a `confidence` score (0–1) alongside its answer. In `aggregate_node`, pick the answer from the specialist with the highest confidence instead of asking the LLM to arbitrate.

---

**Exercise 3 — Trace Which Agent Answered**

Extend `SupervisorState` to include a `routed_to` field. Set it in each specialist node and print it alongside the answer. This is useful for debugging routing errors.

---

**Exercise 4 — Multi-Hop Routing**

Build a two-level hierarchy: a top-level supervisor routes to a **math supervisor** or **science supervisor**, each of which routes to a sub-specialist (e.g., algebra/calculus for math, physics/chemistry for science). Draw the graph before coding it.

---
## Answer Key

Attempt the exercises before reading below.

In [ ]:
# Answer 1 — Creative Writing Specialist
SPECIALISTS_V3 = {
    **SPECIALISTS,
    "creative": "You are a creative writer. Answer using metaphors, imagery, or a short poem.",
}

def supervisor_v3(state: SupervisorState) -> dict:
    r = llm.invoke([
        SystemMessage(content=(
            'Classify into one of: math, history, science, creative. '
            'Respond with ONLY the label.'
        )),
        HumanMessage(content=state["question"]),
    ])
    domain = r.content.strip().lower()
    # guard against unknown labels
    if domain not in SPECIALISTS_V3:
        domain = "science"
    print(f"  [supervisor] -> {domain}")
    return {"next": domain}

g_v3 = StateGraph(SupervisorState)
g_v3.add_node("supervisor", supervisor_v3)
for domain, prompt in SPECIALISTS_V3.items():
    g_v3.add_node(f"{domain}_agent", make_specialist(domain, prompt))
    g_v3.add_edge(f"{domain}_agent", END)
g_v3.add_edge(START, "supervisor")
g_v3.add_conditional_edges("supervisor", lambda s: s["next"],
    {d: f"{d}_agent" for d in SPECIALISTS_V3})
app_v3 = g_v3.compile()

creative_qs = [
    "Explain gravity as a poem.",
    "What is 2 + 2?",
    "Who was Albert Einstein?",
]
for q in creative_qs:
    r = app_v3.invoke({"question": q, "next": "", "answer": ""})
    print(f"Q: {q}")
    print(f"A: {r['answer'][:200]}")
    print()

In [ ]:
# Answer 3 — Trace Which Agent Answered
class TracedState(TypedDict):
    question:  str
    next:      str
    answer:    str
    routed_to: str   # new: tracks which specialist actually ran

def make_traced_specialist(domain: str, prompt: str):
    def node(state: TracedState) -> dict:
        r = llm.invoke([SystemMessage(content=prompt),
                        HumanMessage(content=state["question"])])
        return {"answer": r.content.strip(), "routed_to": domain}
    node.__name__ = f"{domain}_agent"
    return node

gt = StateGraph(TracedState)
gt.add_node("supervisor", lambda s: {"next": supervisor_node(s)["next"]})
for domain, prompt in SPECIALISTS.items():
    gt.add_node(f"{domain}_agent", make_traced_specialist(domain, prompt))
    gt.add_edge(f"{domain}_agent", END)
gt.add_edge(START, "supervisor")
gt.add_conditional_edges("supervisor", lambda s: s["next"],
    {d: f"{d}_agent" for d in SPECIALISTS})
appt = gt.compile()

test_qs = ["What is 7 * 8?", "Who was Napoleon?", "Why is the sky blue?"]
for q in test_qs:
    r = appt.invoke({"question": q, "next": "", "answer": "", "routed_to": ""})
    print(f"Q: {q}")
    print(f"  Routed to: {r['routed_to']}")
    print(f"  A: {r['answer'][:100]}")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*